# Train UNet Refinement — SAM3 → VOC2012 Standard

**Pipeline:**
- Input: RGB (3ch) + SAM3 coarse mask (1ch) + class one-hot (4ch) = **8 channels**
- Output: binary refined mask theo chuẩn VOC
- Target classes: `diningtable`, `sofa`, `chair`, `bicycle`
- Chỉ train trên **positive pairs** (ảnh có class trong GT)
- **Không** dùng data augmentation
- Encoder: **ResNet-34 pretrained** (ImageNet), modified first conv để nhận 8ch

In [1]:
%pip install -q tqdm pillow matplotlib
import os, json, time, random, math, gc
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import torchvision.models as tvm
from torchvision.models import ResNet34_Weights
from torch.utils.data import Dataset, DataLoader

import sam3
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch :', torch.__version__)
print('Device  :', DEVICE)
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


d:\Sam3_voc2012\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Sam3_voc2012\.venv\Lib\site-packages\sam3\model_builder.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


PyTorch : 2.10.0+cu126
Device  : cuda
GPU     : NVIDIA GeForce RTX 3050 Laptop GPU


d:\Sam3_voc2012\.venv\Lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
# ─── Paths ────────────────────────────────────────────────────────────────────
VOC_ROOT         = Path('archive/VOC2012')
JPEG_DIR         = VOC_ROOT / 'JPEGImages'
MASK_DIR         = VOC_ROOT / 'SegmentationClass'
TRAIN_SPLIT_FILE = VOC_ROOT / 'ImageSets' / 'Segmentation' / 'train.txt'

_sam3_dir    = Path(sam3.__file__).parent
BPE_PATH     = _sam3_dir / 'assets' / 'bpe_simple_vocab_16e6.txt.gz'
SAM3_CKPT    = Path(os.getenv('SAM3_CHECKPOINT_PATH', 'sam3.pt')).resolve()

COARSE_CACHE_DIR = Path('coarse_cache')
COARSE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_META_PATH  = COARSE_CACHE_DIR / 'meta.json'

WEIGHTS_DIR = Path('weights')
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
BEST_CKPT    = WEIGHTS_DIR / 'unet_refine_best.pth'
LAST_CKPT    = WEIGHTS_DIR / 'unet_refine_last.pth'
HISTORY_JSON = WEIGHTS_DIR / 'unet_refine_history.json'

# ─── VOC classes ──────────────────────────────────────────────────────────────
VOC_CLASSES = [
    'aeroplane','bicycle','bird','boat','bottle',
    'bus','car','cat','chair','cow',
    'diningtable','dog','horse','motorbike','person',
    'pottedplant','sheep','sofa','train','tvmonitor',
]
VOC_CLASS_TO_IDX = {name: i + 1 for i, name in enumerate(VOC_CLASSES)}

# 4 target classes for UNet refinement
TARGET_CLASSES  = ['diningtable', 'sofa', 'chair', 'bicycle']
TARGET_TO_VOC   = {name: VOC_CLASS_TO_IDX[name] for name in TARGET_CLASSES}
TARGET_TO_LOCAL = {name: i for i, name in enumerate(TARGET_CLASSES)}   # 0-based
LOCAL_TO_TARGET = {i: name for i, name in enumerate(TARGET_CLASSES)}

NUM_CLASSES = len(TARGET_CLASSES)  # 4

# ─── Hyperparameters ──────────────────────────────────────────────────────────
SEED              = 42
IMAGE_SIZE        = 512
BATCH_SIZE        = 4
EPOCHS            = 50
LR                = 1e-4
WEIGHT_DECAY      = 1e-4
WARMUP_EPOCHS     = 3
EARLY_STOP_PATIENCE = 10
REFINE_THRESHOLD  = 0.5          # sigmoid threshold for binary prediction

# SAM3 coarse cache thresholds (try in order, use first that gives ≥1 mask)
COARSE_THRESHOLDS = [0.50, 0.30, 0.20, 0.15]

REBUILD_CACHE = bool(int(os.getenv('REBUILD_CACHE', '0')))
NUM_WORKERS   = 0 if os.name == 'nt' else 4  # Windows Jupyter: 0 workers

# ─── Seed & train/val split ───────────────────────────────────────────────────
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)

with open(TRAIN_SPLIT_FILE, 'r') as f:
    all_ids = [x.strip() for x in f if x.strip()]

rng = random.Random(SEED)
rng.shuffle(all_ids)
split = int(len(all_ids) * 0.8)
train_ids = all_ids[:split]
val_ids   = all_ids[split:]

print(f'Total: {len(all_ids)} | Train: {len(train_ids)} | Val: {len(val_ids)}')
print(f'Target classes: {TARGET_CLASSES}')
print(f'VOC indices   : {TARGET_TO_VOC}')
print(f'SAM3 ckpt     : {SAM3_CKPT}')
print(f'Cache dir     : {COARSE_CACHE_DIR}')


Total: 1464 | Train: 1171 | Val: 293
Target classes: ['diningtable', 'sofa', 'chair', 'bicycle']
VOC indices   : {'diningtable': 11, 'sofa': 18, 'chair': 9, 'bicycle': 2}
SAM3 ckpt     : D:\Sam3_voc2012\sam3.pt
Cache dir     : coarse_cache


In [3]:
# ─── SAM3 Coarse Cache Precomputation ─────────────────────────────────────────
# Chạy SAM3 một lần duy nhất, lưu coarse mask vào .npz để Dataset load nhanh.
# Mỗi file: {image_id}.npz  với keys = class name, value = (H,W) uint8 mask.

def _extract_masks_scores(state):
    if 'masks' not in state or state['masks'] is None or len(state['masks']) == 0:
        return [], []
    masks, scores = [], []
    for i in range(len(state['masks'])):
        masks.append(state['masks'][i].squeeze(0).detach().cpu().numpy().astype(bool))
        scores.append(float(state['scores'][i].item()))
    return masks, scores

def _union_at_threshold(masks, scores, shape_hw, thresholds):
    h, w = shape_hw
    scores_np = np.asarray(scores, dtype=np.float32)
    for thr in thresholds:
        idx = np.where(scores_np >= thr)[0]
        if len(idx) > 0:
            union = np.zeros((h, w), dtype=bool)
            for i in idx:
                union |= masks[i]
            return union.astype(np.uint8), thr
    return np.zeros((h, w), dtype=np.uint8), None

def _safe_cuda_empty_cache():
    if not torch.cuda.is_available():
        return
    try:
        torch.cuda.empty_cache()
    except Exception as e:
        print(f'[warn] torch.cuda.empty_cache failed: {e}')

def _build_sam3_processor():
    if not SAM3_CKPT.exists():
        raise FileNotFoundError(f'SAM3 checkpoint not found: {SAM3_CKPT}')

    model_sam = build_sam3_image_model(
        bpe_path=str(BPE_PATH),
        device='cpu',
        checkpoint_path=str(SAM3_CKPT),
        load_from_HF=False,
    )

    if torch.cuda.is_available():
        try:
            model_sam = model_sam.cuda()
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                print('[warn] CUDA OOM while moving SAM3 to GPU; fallback to CPU for cache build.')
                _safe_cuda_empty_cache()
            else:
                raise

    conf_min = min(COARSE_THRESHOLDS)
    processor = Sam3Processor(model_sam, confidence_threshold=conf_min)
    return model_sam, processor

def build_coarse_cache(image_ids, force_rebuild=False, processor=None):
    missing = [
        img_id for img_id in image_ids
        if force_rebuild or not (COARSE_CACHE_DIR / f'{img_id}.npz').exists()
    ]
    if not missing:
        print(f'Coarse cache complete ({len(image_ids)} images). Skipping.')
        return []

    print(f'Building coarse cache: {len(missing)}/{len(image_ids)} images...')
    own_processor = processor is None
    model_sam = None

    if own_processor:
        model_sam, processor = _build_sam3_processor()

    errors = []
    try:
        for step, image_id in enumerate(tqdm(missing, desc='SAM3 coarse cache'), start=1):
            cache_path = COARSE_CACHE_DIR / f'{image_id}.npz'
            try:
                image = Image.open(JPEG_DIR / f'{image_id}.jpg').convert('RGB')
                state = processor.set_image(image)

                coarse_dict = {}
                for cls_name in TARGET_CLASSES:
                    processor.reset_all_prompts(state)
                    state = processor.set_text_prompt(state=state, prompt=cls_name)
                    masks, scores = _extract_masks_scores(state)
                    union, _ = _union_at_threshold(
                        masks, scores, (image.height, image.width), COARSE_THRESHOLDS
                    )
                    coarse_dict[cls_name] = union

                np.savez_compressed(cache_path, **coarse_dict)
            except Exception as e:
                errors.append((image_id, str(e)))

            if step % 128 == 0:
                gc.collect()
                _safe_cuda_empty_cache()
    finally:
        if own_processor:
            try:
                del processor
            except Exception:
                pass
            if model_sam is not None:
                del model_sam
            gc.collect()
            _safe_cuda_empty_cache()

    print(f'Done. Errors: {len(errors)}')
    for img_id, err in errors[:5]:
        print(f'  {img_id}: {err}')
    return errors

# Build cache cho cả train + val
build_coarse_cache(all_ids, force_rebuild=REBUILD_CACHE)


Building coarse cache: 330/1464 images...


SAM3 coarse cache: 100%|██████████| 330/330 [51:42<00:00,  9.40s/it]


Done. Errors: 0


[]

In [4]:
# Rebuild phần cache còn thiếu hoặc hỏng, không build lại toàn bộ
REQ_KEYS = set(TARGET_CLASSES)

def _cache_ok(image_id):
    p = COARSE_CACHE_DIR / f'{image_id}.npz'
    if not p.exists():
        return False, 'missing_file'
    try:
        with np.load(p) as d:
            keys = set(d.files)
            miss = REQ_KEYS - keys
            if miss:
                return False, f'missing_keys:{sorted(list(miss))}'
            for cls_name in TARGET_CLASSES:
                arr = d[cls_name]
                if arr.ndim != 2:
                    return False, f'bad_ndim:{cls_name}:{arr.ndim}'
            return True, 'ok'
    except Exception as e:
        return False, f'corrupt:{type(e).__name__}:{str(e)[:120]}'

# 1) Audit cache hiện tại
todo_ids = []
todo_reason = {}
for image_id in tqdm(all_ids, desc='Audit coarse_cache'):
    ok, reason = _cache_ok(image_id)
    if not ok:
        todo_ids.append(image_id)
        todo_reason[image_id] = reason

print(f'Need rebuild: {len(todo_ids)}/{len(all_ids)}')
for image_id in todo_ids[:10]:
    print(f'  {image_id}: {todo_reason[image_id]}')

# 2) Rebuild missing/corrupt trong 1 lần để tránh reload model nhiều lần
if len(todo_ids) == 0:
    print('Cache da day du, khong can rebuild.')
else:
    print('Rebuilding missing/corrupt cache in one pass (single SAM3 load)...')
    build_coarse_cache(todo_ids, force_rebuild=True)

# 3) Verify sau rebuild
remain = []
for image_id in all_ids:
    ok, _ = _cache_ok(image_id)
    if not ok:
        remain.append(image_id)

print(f'Cache ready: {len(all_ids) - len(remain)}/{len(all_ids)}')
print(f'Still missing/corrupt: {len(remain)}')
print('Sample remain:', remain[:10])

Audit coarse_cache: 100%|██████████| 1464/1464 [00:11<00:00, 125.78it/s]


Need rebuild: 0/1464
Cache da day du, khong can rebuild.
Cache ready: 1464/1464
Still missing/corrupt: 0
Sample remain: []


In [5]:
# ─── Scan positive pairs & compute per-class pos_weight ───────────────────────
# Positive pair: (image_id, cls_name) khi class đó TỒN TẠI trong GT mask.
# pos_weight được tính CHỈ trên positive images để tránh inflate ratio.

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

TRAIN_POS_PAIRS = set()   # {(image_id, cls_name)}
VAL_POS_PAIRS   = set()

# Per-class pixel counts trên train (chỉ positive images)
cls_pos_px = {cls: 0 for cls in TARGET_CLASSES}
cls_neg_px = {cls: 0 for cls in TARGET_CLASSES}

for image_id in tqdm(train_ids, desc='Scanning train GT masks'):
    mask_path = MASK_DIR / f'{image_id}.png'
    if not mask_path.exists():
        continue
    mask_np = np.array(Image.open(mask_path), dtype=np.uint8)
    valid    = mask_np != 255
    total_valid = int(valid.sum())

    for cls_name in TARGET_CLASSES:
        voc_idx  = TARGET_TO_VOC[cls_name]
        cls_mask = (mask_np == voc_idx) & valid
        p = int(cls_mask.sum())
        if p > 0:
            TRAIN_POS_PAIRS.add((image_id, cls_name))
            cls_pos_px[cls_name] += p
            cls_neg_px[cls_name] += (total_valid - p)

for image_id in tqdm(val_ids, desc='Scanning val GT masks'):
    mask_path = MASK_DIR / f'{image_id}.png'
    if not mask_path.exists():
        continue
    mask_np = np.array(Image.open(mask_path), dtype=np.uint8)
    for cls_name in TARGET_CLASSES:
        voc_idx = TARGET_TO_VOC[cls_name]
        if (mask_np == voc_idx).any():
            VAL_POS_PAIRS.add((image_id, cls_name))

# Per-class BCE pos_weight: neg/pos ratio, clipped [1, 50]
CLS_POS_WEIGHTS = {}
for cls_name in TARGET_CLASSES:
    ratio = cls_neg_px[cls_name] / max(cls_pos_px[cls_name], 1)
    CLS_POS_WEIGHTS[cls_name] = float(np.clip(ratio, 1.0, 50.0))
CLS_POS_WEIGHT_TENSOR = torch.tensor(
    [CLS_POS_WEIGHTS[c] for c in TARGET_CLASSES],
    dtype=torch.float32, device=DEVICE
)

print(f'\nTrain positive pairs: {len(TRAIN_POS_PAIRS)}')
print(f'Val   positive pairs: {len(VAL_POS_PAIRS)}')
print(f'\nPer-class stats (train positive images only):')
for cls_name in TARGET_CLASSES:
    local_id = TARGET_TO_LOCAL[cls_name]
    print(f'  {cls_name:11s} | pos_px={cls_pos_px[cls_name]:,} '
          f'| neg_px={cls_neg_px[cls_name]:,} '
          f'| pos_weight={CLS_POS_WEIGHTS[cls_name]:.2f}')


Scanning val GT masks: 100%|██████████| 293/293 [00:02<00:00, 116.26it/s]


Train positive pairs: 323
Val   positive pairs: 65

Per-class stats (train positive images only):
  diningtable | pos_px=2,895,069 | neg_px=8,861,594 | pos_weight=3.06
  sofa        | pos_px=2,740,198 | neg_px=10,506,850 | pos_weight=3.83
  chair       | pos_px=2,272,705 | neg_px=18,056,119 | pos_weight=7.94
  bicycle     | pos_px=523,826 | neg_px=8,559,531 | pos_weight=16.34
